In [1]:
# Install TimeSynth if not installed already
%pip install git+https://github.com/TimeSynth/TimeSynth.git

  Cloning https://github.com/TimeSynth/TimeSynth.git to /tmp/pip-req-build-in24m8v9
  Running command git clone --filter=blob:none --quiet https://github.com/TimeSynth/TimeSynth.git /tmp/pip-req-build-in24m8v9
  Resolved https://github.com/TimeSynth/TimeSynth.git to commit e50cdb9015d415adf46a4eae161a087c5c378564
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.2/136.2 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 20.3 MB/s eta 0:00:00
  Created wheel for timesynth: filename=timesynth-0.2.4-py3-none-any.whl size=15422 sha256=9ef98e1e64de3c02d362d3a5b53b384b4356a88977a36bed7ac9bc22a9c8a17e
  Stored in directory: /tmp/pip-ephem-wheel-cache-tedp3ya_/wheels/cb/ef/be/19ae3d8020210aede9ae0d9eec80fc4797ff33b4513bdf3641
  Created wheel for jitcdde: filename=jitcdde-1.4.0-py3-none-any.whl size=30854 sha256=2f698132ac49ce12fd5138ba6edaeeb2c

In [42]:
import numpy as np
import matplotlib.pyplot as plt
import timesynth as ts
import pandas as pd
np.random.seed(123)


# Data Generating Process and Synthetic Time Series

In [25]:
def plot_time_series(time, values, label, legends=None):
    # Check if multiple time series are provided
    if legends is not None:
        # Ensure the number of legends matches the number of value arrays
        assert len(legends)==len(values)

    # Handle cases for multiple or single time series for plotting
    if isinstance(values, list):
        # Create a dictionary to hold time and multiple value series
        series_dict = {"Time": time}
        # Populate the dictionary with each series and its legend
        for v, l in zip(values, legends):
            series_dict[l] = v
        # Convert the dictionary to a pandas DataFrame
        plot_df = pd.DataFrame(series_dict)
        # Melt the DataFrame to a 'long' format, suitable for plotting multiple lines with plotly.express
        plot_df = pd.melt(plot_df,id_vars="Time",var_name="ts", value_name="Value")
    else:
        # If only a single series, create a DataFrame directly
        series_dict = {"Time": time, "Value": values, "ts":""}
        plot_df = pd.DataFrame(series_dict)

    # Generate the line plot using plotly.express
    if isinstance(values, list):
        # For multiple series, use 'ts' column to differentiate lines (e.g., by dash style)
        fig = px.line(plot_df, x="Time", y="Value", line_dash="ts")
    else:
        # For a single series, a simple line plot is sufficient
        fig = px.line(plot_df, x="Time", y="Value")

    # Customize the plot layout for better aesthetics and readability
    fig.update_layout(
        autosize=False,
        width=900,
        height=500,
        title={
        'text': label,
#         'y':0.9,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'},
        titlefont={
            "size": 25
        },
        yaxis=dict(
            title_text="Value",
            titlefont=dict(size=12),
        ),
        xaxis=dict(
            title_text="Time",
            titlefont=dict(size=12),
        )
    )
    return fig

def generate_timeseries(signal, noise=None):
    # Initialize a TimeSampler to define the time points for the series
    # This creates a regular time axis from t=0 to t=20 with 100 points
    time_sampler = ts.TimeSampler(stop_time=20)
    regular_time_samples = time_sampler.sample_regular_time(num_points=100)

    # Create a TimeSeries object using the provided signal and optional noise generators
    timeseries = ts.TimeSeries(signal_generator=signal, noise_generator=noise)

    # Sample the time series to get the actual data points
    # Returns: samples (signal + noise), signals (pure signal), errors (noise component)
    samples, signals, errors = timeseries.sample(regular_time_samples)

    # Return the generated data and time axis
    return samples, regular_time_samples, signals, errors

## White Noise

In [26]:
# Generate the time axis with sequential numbers upto 200
time = np.arange(200)
# Sample 200 hundred random values
values = np.random.randn(200)*100
fig = plot_time_series(time, values, "")
fig.show()

## Red Noise

In [27]:
# Setting the correlation coefficient
r = 0.4
# Generate the time axis
time = np.arange(200)
# Generate white noise
white_noise = np.random.randn(200)*100
# Create Red Noise by introducing correlation between subsequent values in the white noise
values = np.zeros(200)
for i, v in enumerate(white_noise):
    if i==0:
        values[i] = v
    else:
        values[i] = r*values[i-1]+ np.sqrt((1-np.power(r,2))) *v


fig = plot_time_series(time, values, "")
fig.show()

## Sinusoidal

In [28]:
#Sinusoidal Signal with Amplitude=1.5 & Frequency=0.25
signal_1 =ts.signals.Sinusoidal(amplitude=1.5, frequency=0.25)
#Sinusoidal Signal with Amplitude=1 & Frequency=0. 5
signal_2 = ts.signals.Sinusoidal(amplitude=1, frequency=0.5)
#Generating the time series
samples_1, regular_time_samples, signals_1, errors_1 = generate_timeseries(signal=signal_1)
samples_2, regular_time_samples, signals_2, errors_2 = generate_timeseries(signal=signal_2)

In [29]:
fig = plot_time_series(regular_time_samples,
                 [samples_1, samples_2],
                 "",
                 legends=["Amplitude = 1.5 | Frequency = 0.25", "Amplitude = 1 | Frequency = 0.5"])
fig.show()

## Pseudo Periodic

In [30]:
# PseudoPeriodic signal with Amplitude=1 & Frequency=0.25
signal = ts.signals.PseudoPeriodic(amplitude=1, frequency=0.25)
#Generating Timeseries
samples, regular_time_samples, signals, errors = generate_timeseries(signal=signal)



In [31]:
fig = plot_time_series(regular_time_samples,
                 samples,
                 "")
fig.show()

## Auto Regressive

In [32]:
# TimeSynth package has an open issue on its autoregressive function.
# As of 3/16/24 the issue is still open.
# Link to issue https://github.com/TimeSynth/TimeSynth/issues/23
# Thus below is the rebuilt function with fix.

class BaseSignal:
    """BaseSignal class

    Signature for all signal classes.

    """

    def __init__(self):
        # Constructor for the Abstract BaseSignal class. Raises an error to ensure it's not instantiated directly.
        raise NotImplementedError

    def sample_next(self, time, samples, errors):
        """Samples next point based on history of samples and errors

        Parameters
        ----------
        time : int
            time
        samples : array-like
            all samples taken so far
        errors : array-like
            all errors sampled so far

        Returns
        -------
        float
            sampled signal for time t

        """
        # This method should be implemented by subclasses to generate the next sample.
        raise NotImplementedError

    def sample_vectorized(self, time_vector):
        """Samples for all time points in input

        Parameters
        ----------
        time_vector : array like
            all time stamps to be sampled

        Returns
        -------
        float
            sampled signal for time t

        """
        # This method should be implemented by subclasses for vectorized sampling.
        raise NotImplementedError


__all__ = ['AutoRegressive']


class AutoRegressive(BaseSignal):
    """Sample generator for autoregressive (AR) signals.

    Generates time series with an autogressive lag defined by the number of parameters in ar_param.
    NOTE: Only use this for regularly sampled signals

    Parameters
    ----------
    ar_param : list (default [None])
        Parameter of the AR(p) process
        [phi_1, phi_2, phi_3, .... phi_p]
    sigma : float (default 1.0)
        Standard deviation of the signal
    start_value : list (default [None])
        Starting value of the AR(p) process

    """

    def __init__(self, ar_param=[None], sigma=0.5, start_value=[None]):
        # Indicates that this signal generator is not designed for vectorized sampling
        self.vectorizable = False
        # Reverse the AR parameters for easier calculation later (phi_p, ..., phi_1)
        ar_param.reverse()
        # Store the AR parameters
        self.ar_param = ar_param
        # Store the standard deviation for the noise
        self.sigma = sigma
        # Initialize starting values for the AR process
        if start_value[0] is None:
            # If no start values are provided, default to zeros based on the number of AR parameters
            self.start_value = [0 for i in range(len(ar_param))]
        else:
            # If start values are provided, validate their length
            if len(start_value) != len(ar_param):
                raise ValueError("AR parameters do not match starting value")
            else:
                self.start_value = start_value
        # Initialize previous_value with the start_value for the first calculation
        self.previous_value = self.start_value

    def sample_next(self, time, samples, errors):
        """Sample a single time point

        Parameters
        ----------
        time : number
            Time at which a sample was required

        Returns
        -------
        ar_value : float
            sampled signal for time t
        """
        # Calculate the autoregressive component: sum of (previous_value * ar_param)
        ar_value = sum(self.previous_value[i] * self.ar_param[i] for i in range(len(self.ar_param)))
        # Generate a random noise term from a normal distribution
        noise = np.random.normal(loc=0.0, scale=self.sigma)
        # Add the noise to the autoregressive value
        ar_value += noise
        # Update the list of previous values by dropping the oldest and adding the new ar_value
        self.previous_value = self.previous_value[1:] + [ar_value]
        # Return the newly sampled autoregressive value
        return ar_value

In [33]:
# y(t) = 1.5*y(t-1) - 0.75*y(t-2)
signal=AutoRegressive(ar_param=[1.5, -0.75])
#Generate Timeseries
samples, regular_time_samples, signals, errors = generate_timeseries(signal=signal)



In [34]:
fig = plot_time_series(regular_time_samples,
                 samples,
                 "")
fig.show()

## Mix and Match

In [35]:
#Generating Pseudo Periodic Signal
pseudo_samples, regular_time_samples, _, _ = generate_timeseries(signal=ts.signals.PseudoPeriodic(amplitude=1, frequency=0.25), noise=ts.noise.GaussianNoise(std=0.3))
# Generating an Autoregressive Signal
ar_samples, regular_time_samples, _, _ = generate_timeseries(signal=AutoRegressive(ar_param=[1.5, -0.75]))
# Combining the two signals using a mathematical equation
timeseries_ = pseudo_samples*2+ar_samples

In [36]:
fig = plot_time_series(regular_time_samples,
                 timeseries_,
                 "")
fig.show()

## Non-Stationary: Sinusoidal with Trend and White Noise

In [37]:
# Sinusoidal Signal with Amplitude=1 & Frequency=0.25
signal=ts.signals.Sinusoidal(amplitude=1, frequency=0.25)
# White Noise with standar deviation = 0.3
noise=ts.noise.GaussianNoise(std=0.3)
# Generate the time series
sinusoidal_samples, regular_time_samples, _, _ = generate_timeseries(signal=signal, noise=noise)
# Regular_time_samples is a linear incteasing time axis and can be used as a trend
trend = regular_time_samples*0.4
# Combining the signal and trend
timeseries_ = sinusoidal_samples+trend

In [38]:
fig = plot_time_series(regular_time_samples,
                 timeseries_,
                 "")
fig.show()

## Non-Stationary: Sinusoidal and Time Varying Noise

In [39]:
sinusoidal_samples, regular_time_samples, _, _ = generate_timeseries(signal=ts.signals.Sinusoidal(amplitude=1, frequency=0.25))

In [40]:
noise = [np.random.randn()*np.sqrt(i) for i, v in enumerate(regular_time_samples)]

In [41]:
fig = plot_time_series(regular_time_samples,
                 sinusoidal_samples+noise,
                 "")
fig.show()